# Índice FIBRAS de AMEFIBRA

Notebook para extraer la tabla pública del Índice FIBRAS. La página carga los datos dentro de un `iframe` mediante JavaScript y WebSocket, por lo que se utiliza Playwright con Chromium.

> La información se ofrece únicamente para consulta y análisis. AMEFIBRA indica que los datos tienen aproximadamente 20 minutos de retraso y no deben usarse como base única para decisiones de inversión.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

try:
    # VS Code inyecta esta variable con la ruta absoluta del propio notebook,
    # así que la raíz del proyecto queda anclada a dónde vive el archivo .ipynb,
    # sin importar cuál sea el directorio de trabajo con el que arrancó el kernel
    # (que puede no ser la raíz del proyecto, según la configuración del editor).
    RAIZ_PROYECTO = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    RAIZ_PROYECTO = Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from modules.presentacion import (
    aplicar_tema_oscuro_notebook,
    armar_tabla_tickers_seleccionados,
    descargar_historiales_dividendos,
    ejecutar_extraccion_indice,
    exportar_csv_excel,
    exportar_xlsx,
    mostrar_emisoras,
    mostrar_fichas_completas_cliente_multi,
    mostrar_fichas_rendimiento_multi,
    resumen_precio_periodicidad,
    seleccionar_anio_interactivo,
    seleccionar_tickers_interactivo,
)
from modules.procesamiento import obtener_anios_disponibles_comunes

In [2]:
# Aplicar tema oscuro al notebook
aplicar_tema_oscuro_notebook()

# Configuración de rutas de salida
CARPETA_SALIDA = Path.cwd() / "output"
CARPETA_FICHAS_PDF = CARPETA_SALIDA / "fichas"

# Parámetros editables del notebook
HEADLESS = True
TIMEOUT_DATOS_MS = 30000
EXPORTAR_CSV_ANALITICO = True
EXPORTAR_CSV_EXCEL = False
EXPORTAR_XLSX = False
RUTA_CSV_EXCEL = Path.cwd() / "indice_fibras.csv"
RUTA_XLSX = Path.cwd() / "indice_fibras.xlsx"

## Ejecutar extracción
Consultar solo los lunes temprano para hacer un análisis rápido de las FIBRAS y para saber si hubo altas y bajas de emisoras.

In [3]:
df = ejecutar_extraccion_indice(HEADLESS, TIMEOUT_DATOS_MS, CARPETA_SALIDA, EXPORTAR_CSV_ANALITICO)

Consultando https://amefibra.com/el-mercado/indice-fibras/ ...
Índice FIBRAS - 2026-09-02 13:38 (dato con ~20 min de retraso)


,Emisora,Cotización,Var.,Var. %,Apertura,Máx. día,Min. día,Promedio,Operaciones,Volumen,Importe,Máx. 52 s.,Min. 52 s.
0,DANHOS13,29.10,-0.12,-0.41%,29.11,29.30,29.30,28.91,1080,75055,2184190,29.22,23.94
1,EDUCA18,54.00,0.00,0.00%,0.00,-,-,-,4,17,901,57.63,45.81
2,FIBRAMQ12,43.74,-0.31,-0.70%,43.97,44.24,44.24,43.70,340,27492,1204894,45.06,27.60
3,FIBRAPL14,75.52,-0.50,-0.66%,75.35,75.63,75.95,74.75,2611,934010,70449226,83.97,64.87
4,FIBRAUP18,37.50,0.00,0.00%,0.00,-,-,-,3,3,113,41.00,17.27
5,FIHO12,7.62,0.01,0.13%,7.62,7.64,7.65,7.58,231,13584,103544,8.01,6.92
6,FINN13,4.80,0.06,1.27%,4.79,4.78,4.80,4.77,173,7484,35818,5.40,4.33
7,FMTY14,14.27,-0.03,-0.21%,14.27,14.37,14.39,14.15,7478,1198446,17067159,15.70,12.37
8,FNOVA17,41.90,0.00,0.00%,41.65,41.60,42.00,41.30,248,24790,1038150,45.95,27.00
9,FPLUS16,5.05,0.03,0.60%,5.00,5.05,5.05,4.96,327,21569,107631,6.00,4.82


CSV analítico guardado en: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260902_133858_indice_fibras_amefibra.csv


### Exportar a archivo de Excel - xlsx (Ejecución opcional)

In [4]:
if EXPORTAR_CSV_EXCEL:
    ruta_csv_excel = exportar_csv_excel(df, RUTA_CSV_EXCEL)
    print(f"CSV compatible con Excel guardado en: {ruta_csv_excel}")

if EXPORTAR_XLSX:
    ruta_xlsx = exportar_xlsx(df, RUTA_XLSX)
    print(f"Excel guardado en: {ruta_xlsx}")

## Consulta de emisoras

In [5]:
try:
    df
except NameError:
    df = None

df_emisoras = mostrar_emisoras(df, CARPETA_SALIDA)

Fuente de emisoras: extracción de AMEFIBRA de esta corrida.
      Emisora
0    DANHOS13
1     EDUCA18
2   FIBRAMQ12
3   FIBRAPL14
4   FIBRAUP18
5      FIHO12
6      FINN13
7      FMTY14
8     FNOVA17
9     FPLUS16
10    FSHOP13
11     FUNO11
12     NEXT25
13     SOMA21
14  STORAGE18
CSV de emisoras guardado en: d:\devs\dev-workbench\own-projects\tool-python-extract-amefibra-data-fibras\output\20260902_133912_list_of_tickers.csv


## Historial de distribuciones por FIBRA

### Fuentes evaluadas

| Fuente | Cobertura BMV | Datos de distribuciones | Acceso y límites |
|---|---|---|---|
| Relación con Inversionistas del emisor | Sí, por emisora | Fuente primaria; puede incluir fechas, importe y componentes fiscales en PDF/XLSX | Gratuita, sin API uniforme; requiere localizar y procesar reportes de cada emisor |
| AMEFIBRA | Sí, índice agregado | Cotización e indicadores del índice; no publica aquí un histórico normalizado de distribuciones | Consulta web pública; no se expone una API de dividendos en esta tabla |
| BMV/BIVA | Sí | Información oficial de emisoras y eventos, según disponibilidad del portal | Consulta pública, pero sin una API gratuita y estable para este flujo |
| FMP, Alpha Vantage, Twelve Data, EODHD, Nasdaq Data Link y Polygon | Cobertura mexicana variable | La cobertura y profundidad de dividendos para tickers BMV no está garantizada en el plan gratuito | Requieren revisar ticker, API key y límites por proveedor |
| `yfinance` | Sí para tickers Yahoo con sufijo `.MX`, cuando Yahoo dispone del evento | Fecha ex-dividendo y monto; no garantiza fecha de registro, pago ni componentes fiscales | Gratis y sin API key, pero es un cliente no oficial de Yahoo Finance y está sujeto a cambios y límites |

Se usa `yfinance` como respaldo reproducible porque las fuentes primarias no ofrecen una API homogénea. El resultado contiene la fecha ex-dividendo y el importe disponible en Yahoo; la fecha de registro, fecha de pago y componentes fiscales no se incluyen porque esta fuente no los entrega de forma confiable. El histórico se ordena del más antiguo al más reciente. `yield_pct` es el rendimiento de cada distribución respecto al cierre de su fecha ex-dividendo; `annualized_yield_pct` anualiza ese rendimiento usando `365 / días_del_periodo`. Para la primera fila se usa la mediana histórica de días entre distribuciones.

### Tickers a consultar

Elige uno o más tickers en el widget de selección múltiple (mismo listado de la sección "Consulta de emisoras"): clic para uno, Ctrl+clic o Shift+clic para varios. Al correr esta celda se despliega el selector; ajusta la selección y luego corre la celda de abajo para consultar los tickers elegidos.

In [ ]:
selector_tickers = seleccionar_tickers_interactivo(df_emisoras["Emisora"])

In [ ]:
# Armamos el DataFrame de tickers seleccionados (con la cotización de AMEFIBRA de esta
# corrida como metadato, si se ejecutó la extracción). Sobre él descargamos el historial
# de dividendos de cada ticker y mostramos precio actual y periodicidad en una sola tabla.
try:
    df
except NameError:
    df = None

tickers_seleccionados = armar_tabla_tickers_seleccionados(selector_tickers.value, df_emisoras["Emisora"], df)
display(tickers_seleccionados)

historiales = descargar_historiales_dividendos(tickers_seleccionados["ticker"], df_emisoras["Emisora"], CARPETA_SALIDA)
resumen_tickers = resumen_precio_periodicidad(tickers_seleccionados, historiales)

## FICHA DE RENDIMIENTO ANUAL PERSONALIZADO

La ficha usa el **año calendario** (`1 de enero` a `31 de diciembre`). Los pagos se filtran por `ex_date`, que es la fecha disponible en el historial de `yfinance`; no se inventa una fecha de pago que la fuente no proporciona. Los precios inicial y final son el primer y último cierre disponible dentro del año. El rendimiento por dividendos se calcula contra el precio inicial, y el rendimiento de capital contra la variación entre precio final e inicial. La ficha es informativa y no constituye una recomendación de inversión.

### Año a consultar

Elige, del desplegable, el año a consultar. Solo se muestran los años con distribuciones disponibles para **todos** los tickers seleccionados (intersección); el año elegido se usa para la ficha de cada ticker.

In [ ]:
AÑOS_DISPONIBLES = obtener_anios_disponibles_comunes(historiales)
selector_anio = seleccionar_anio_interactivo(AÑOS_DISPONIBLES)

In [ ]:
# Generamos y exportamos (HTML + PDF) la ficha de rendimiento anual de cada ticker
# seleccionado, para el año elegido arriba.
AÑO_SELECCIONADO = selector_anio.value
rutas_fichas_rendimiento_anual = mostrar_fichas_rendimiento_multi(
    tickers_seleccionados, historiales, CARPETA_SALIDA, CARPETA_FICHAS_PDF, año=AÑO_SELECCIONADO
)

### Ficha completa del año seleccionado para cliente

Ficha completa anual pensada como entregable final para el cliente (escenario de inversión, distribuciones mensuales y rendimiento total en el año), con un diseño distinto al de la ficha de rendimiento anterior. Se genera una por cada ticker seleccionado, con el año ya elegido arriba y los datos reales de cada uno; no inventa cifras. Es informativa y no constituye una recomendación de inversión.

In [ ]:
# Generamos y exportamos (HTML + PDF) la ficha completa anual para el cliente de cada
# ticker seleccionado, con el mismo año.
rutas_fichas_completas_anual = mostrar_fichas_completas_cliente_multi(
    tickers_seleccionados, historiales, CARPETA_SALIDA, CARPETA_FICHAS_PDF, año=AÑO_SELECCIONADO
)

## FICHA DE RENDIMIENTO Y RIESGO DE LOS ÚLTIMOS 12 MESES

Misma ficha de rendimiento de arriba, pero calculada sobre la ventana móvil de los últimos 12 meses completos (en vez de año calendario), con el riesgo mensual promedio del periodo (volatilidad del retorno total mensual: variación de precio + dividendos del mes) agregado como cifra destacada junto al rendimiento total.

In [ ]:
# Fecha de referencia para la ventana móvil de 12 meses (fecha_fin del periodo).
# None = usa la fecha actual; fijar una fecha (ej. "2025-12-31") permite correr el
# análisis de forma retrospectiva, útil para pruebas. La ventana es única y se aplica
# por igual a todos los tickers seleccionados. El flujo de año calendario de las celdas
# anteriores no se modifica y sigue disponible como antes.
from modules.procesamiento import calcular_ventana_movil_12_meses

FECHA_REFERENCIA_12M = None
FECHA_INICIO_12M, FECHA_FIN_12M = calcular_ventana_movil_12_meses(FECHA_REFERENCIA_12M)
print(f"Ventana de análisis (única para todos los tickers seleccionados): {FECHA_INICIO_12M:%Y-%m-%d} a {FECHA_FIN_12M:%Y-%m-%d}")

### Ficha sencilla de los últimos 12 meses

In [ ]:
# Generamos y exportamos (HTML + PDF) la ficha sencilla de rendimiento de los últimos
# 12 meses de cada ticker seleccionado, con la ventana definida arriba.
rutas_fichas_rendimiento_12m = mostrar_fichas_rendimiento_multi(
    tickers_seleccionados, historiales, CARPETA_SALIDA, CARPETA_FICHAS_PDF, fecha_referencia=FECHA_REFERENCIA_12M
)

### Ficha completa de los últimos 12 meses para cliente

Misma ficha completa de arriba, pero sobre la ventana móvil de últimos 12 meses: agrega una sección de riesgo del periodo con la volatilidad anualizada del retorno total mensual y la serie de los 12 retornos mensuales en barras (verde = mes positivo, rojo = mes negativo), colocada junto al desglose de rendimiento (plusvalía vs. distribuciones).

In [ ]:
# Generamos y exportamos (HTML + PDF) la ficha completa de los últimos 12 meses para el
# cliente de cada ticker seleccionado.
rutas_fichas_completas_12m = mostrar_fichas_completas_cliente_multi(
    tickers_seleccionados, historiales, CARPETA_SALIDA, CARPETA_FICHAS_PDF, fecha_referencia=FECHA_REFERENCIA_12M
)

## ANÁLISIS DE TRES AÑOS